### Getting_Started 

This notebook gives an explainer of the functionalities of the repository, how to call different functions and parameters that can be changed.

In [ ]:
# --- 1. Imports, environment, and source-file checks ---

import gc
import inspect
import os
import time

# Must be set before importing JAX, WDM, FEW, or anything that may import JAX.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import WDM
import numpy as np
import matplotlib.pyplot as plt

try:
    import psutil
except ImportError:
    psutil = None

from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    one_arm_response_td,
)
from WDM_LISAresponse.code.lisa_response.TDI_with_one_arm_plan import (
    gen2_michelson_td,
    precompute_gen2_michelson_wdm_geometry,
    precompute_gen2_michelson_wdm_waveform,
    gen2_michelson_wdm_from_waveform_precomputed,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
    ONE_ARM_EXECUTION_MODES,
)
from WDM_LISAresponse.code.utils.utils import (
    apply_wdm_cut,
    compute_wdm_cut_info,
    wdm_mismatch,
)

from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits
from few.waveform import GenerateEMRIWaveform
from few.utils.constants import YRSID_SI


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    """Current process resident memory, including JAX/XLA CPU buffers."""
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


def block_result(result):
    """Synchronize JAX arrays before stopping a benchmark timer."""
    if isinstance(result, dict):
        for value in result.values():
            block_result(value)
        return

    if isinstance(result, (tuple, list)):
        for value in result:
            block_result(value)
        return

    if hasattr(result, "block_until_ready"):
        result.block_until_ready()


def median_runtime(callable_, repeats=3):
    """Warm once, then return the median synchronized runtime."""
    result = callable_()
    block_result(result)

    timings = []

    for _ in range(repeats):
        started = time.perf_counter()

        result = callable_()
        block_result(result)

        timings.append(time.perf_counter() - started)

    return float(np.median(timings))


print("Requested JAX platforms:", os.environ["JAX_PLATFORMS"])
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())

assert jax.default_backend() == "cpu", (
    "JAX was not initialized on CPU. Restart the kernel and run this "
    "cell before any JAX-related import."
)
assert all(device.platform == "cpu" for device in jax.devices())

Setup of the lisaorbits functions and the EMRI waveform generation functions

In [ ]:
#very expensive to build td_gen 

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True
)

orbit_data = read_orbit_file('esa-trailing-orbits.h5')

orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
t_start = orbit_data.t0

orbits_trailing = InterpolatedOrbits(orbit_times, 
                                     orbit_data.positions, 
                                     orbit_data.velocities)

Parameters for the waveform generation

In [ ]:

#window function parameters 
alpha = 0.005
tukey_factor = 4

#data series parameters
t0 = t_start + 1.0*YRSID_SI
N = 3200000
dt = 10
Nf = 100
#T_obs = 1 #can use either N or T_obs 

#orbit parameters for EMRI
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0

#sky position parameters
dist = 1.0
theta = np.pi / 2
phi = 0
qK = np.pi / 3
phiK = np.pi / 3

#accuracy of interpolation
eps = 1e-2

# initial phases
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3

### Numerical and benchmark settings

The WDM calculation uses the staged API:

- **A0:** WDM grid, one-arm geometry, one-arm shift plan, recursive TDI delays, and the prepared 42-job TDI plan;
- **A1:** transform the actual waveform and apply the prepared six-link one-arm plan;
- **B:** apply the prepared TDI plan and accumulate the second-generation Michelson \(X,Y,Z\) channels.

For long laptop runs, grouped TDI execution is the production path. The fully materialized 42-job route is retained only as an optional numerical and scaling reference because its temporary output exceeds \(1\) GiB at approximately one-year duration.


In [ ]:

q_wdm = 4
L_trunc_value = 25
num_int_points = 20

# Long-duration notebook controls.
RUN_TD_REFERENCE = False
RUN_MATERIALIZED_REFERENCE = False
BENCHMARK_REPEATS = 9

# One-arm validation and comparison controls.
RUN_ONE_ARM_VALIDATION = True
RUN_ONE_ARM_BENCHMARK = True
ONE_ARM_BENCHMARK_REPEATS = 15
FULL_A1_BENCHMARK_REPEATS = 9

# Selected full-year CPU route.
PARALLEL_WDM_TRANSFORMS = True
ONE_ARM_EXECUTION_MODE = "materialized_parallel_links"
ONE_ARM_PARALLEL_LINKS_PER_GROUP = 3
ONE_ARM_PARALLEL_MAX_WORKERS = 2

fast_shift_kwargs = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "row_chunk_size": 32640,
    "lag_block_size": 17,
    "job_block_size": 1,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
}

faithful_shift_kwargs = {
    "assembly_backend": "lagfirst_chunked",
    "assembly_precision": "complex128",
    "row_chunk_size": 128,
    "tl_tp_mode": "exact",
    "use_jax": True,
}

legacy_shift_kwargs = {
    "assembly_backend": "legacy",
    "assembly_precision": "complex128",
    "tl_tp_mode": "exact",
    "use_jax": True,
}

# Choose the operator configuration during A0. Do not override it in A1.
shift_kwargs = fast_shift_kwargs

print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("Parallel WDM transforms:", PARALLEL_WDM_TRANSFORMS)
print("One-arm execution mode:", ONE_ARM_EXECUTION_MODE)
print(
    "One-arm link schedule:",
    f"{6 // ONE_ARM_PARALLEL_LINKS_PER_GROUP} groups × "
    f"{ONE_ARM_PARALLEL_LINKS_PER_GROUP} links",
)
print(f"Process RSS before waveform generation: {rss_gib():.2f} GiB")


Waveform generation + plotting 

In [ ]:
model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model="emri",
    t0=t0,
    dt=dt,
    N= N,
    model_kwargs=model_kwargs,
    window_function="tukey",
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
 )

print(f"Generated padded waveform length: {len(times_pad)}")
print(f"WDM Nt x Nf = {pad_info['wdm_Nt']} x {Nf}")
print(f"Tail right padding: {pad_info['wdm_tail_pad_right']}")

plotting_times = times_pad - times_pad[0]
plt_left_lim = int(0.5 * (len(times_pad) - N))

plt.figure(figsize=(10, 4))
plt.plot(plotting_times[plt_left_lim:plt_left_lim+4000], hplus_td[plt_left_lim:plt_left_lim+4000], label="h_plus")
plt.plot(plotting_times[plt_left_lim:plt_left_lim+4000], hcross_td[plt_left_lim:plt_left_lim+4000], label="h_cross", alpha=0.8)
plt.title("EMRI waveform (first 2000 samples)")
plt.xlabel("time [s]")
plt.ylabel("strain")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print(f"Process RSS after waveform generation: {rss_gib():.2f} GiB")



### Time-domain one-arm reference


In [ ]:
if RUN_TD_REFERENCE:
    t_td0 = time.perf_counter()

    dl_12_td = one_arm_response_td(
        times_pad,
        1,
        2,
        orbits_trailing,
        theta,
        phi,
        hplus_td,
        hcross_td,
        num_int_points=num_int_points,
    )

    td_one_arm_time = time.perf_counter() - t_td0

    print(f"TD one-arm runtime: {td_one_arm_time:.3f} s")
    print(f"TD one-arm shape: {dl_12_td.shape}")
else:
    dl_12_td = None
    print("TD one-arm reference skipped. Set RUN_TD_REFERENCE=True to run it.")


In [ ]:
# --------------------
# A0: static/grid/geometry/operator setup
# --------------------
t_a0 = time.perf_counter()

geom_precomp = precompute_gen2_michelson_wdm_geometry(
    times_pad,
    theta,
    phi,
    orbits_trailing,
    Nf=Nf,
    L_trunc=L_trunc_value,
    q=q_wdm,
    d=4,
    A_frac=0.25,
    calc_m0=True,
    variable_shift_kwargs=shift_kwargs,
)

a0_time = time.perf_counter() - t_a0
rss_after_a0 = rss_gib()

one_arm_shift_plan = geom_precomp["one_arm_plan"].shift_plan
tdi_plan = geom_precomp["tdi_plan"]

assert (
    one_arm_shift_plan.resolved_assembly_backend
    == "lagfirst_chunked_lagblock"
), one_arm_shift_plan.resolved_assembly_backend

assert (
    tdi_plan.shift_plan.resolved_assembly_backend
    == "lagfirst_chunked_lagblock"
), tdi_plan.shift_plan.resolved_assembly_backend

assert hasattr(
    one_arm_shift_plan,
    "apply_weighted_sources_device",
), (
    "The loaded VariableShiftBatchPlan does not contain the commit-3B "
    "weighted-source method. Restart the kernel after replacing the source files."
)

# --------------------
# A1: actual waveform -> six one-arm WDM links
# --------------------
# Keep the six shifted links on-device while selecting the faster
# device-materialized one-arm implementation explicitly.
t_a1 = time.perf_counter()

wave_precomp = precompute_gen2_michelson_wdm_waveform(
    geom_precomp,
    hplus_td,
    hcross_td,
    return_device_links=True,
    parallel_wdm_transforms=PARALLEL_WDM_TRANSFORMS,
    one_arm_execution_mode=ONE_ARM_EXECUTION_MODE,
    one_arm_parallel_links_per_group=(
        ONE_ARM_PARALLEL_LINKS_PER_GROUP
    ),
    one_arm_parallel_max_workers=(
        ONE_ARM_PARALLEL_MAX_WORKERS
    ),
)

# JAX execution is asynchronous. Synchronize all links before stopping
# the A1 timer, otherwise this records dispatch rather than runtime.
block_result(tuple(wave_precomp["link_w"].values()))

a1_time = time.perf_counter() - t_a1
rss_after_a1 = rss_gib()

assert wave_precomp["one_arm_plan"] is geom_precomp["one_arm_plan"]
assert wave_precomp["tdi_plan"] is geom_precomp["tdi_plan"]
assert wave_precomp["one_arm_execution_mode"] == ONE_ARM_EXECUTION_MODE
assert wave_precomp["parallel_wdm_transforms"] is PARALLEL_WDM_TRANSFORMS
assert (
    wave_precomp["one_arm_parallel_links_per_group"]
    == ONE_ARM_PARALLEL_LINKS_PER_GROUP
)
assert (
    wave_precomp["one_arm_parallel_max_workers"]
    == ONE_ARM_PARALLEL_MAX_WORKERS
)

wdm_response = wave_precomp["wdm"]
link_w = wave_precomp["link_w"]
w_12_wdm = link_w[(1, 2)]

for link_array in link_w.values():
    assert isinstance(link_array, jax.Array)

a0_accounted = (
    geom_precomp["wdm_grid_build_s"]
    + geom_precomp["one_arm_plan_build_s"]
    + geom_precomp["tdi_delay_build_s"]
    + geom_precomp["tdi_plan_build_s"]
)

waveform_context_s = float(wave_precomp["waveform_context_build_s"])
synchronized_link_stage_s = max(0.0, a1_time - waveform_context_s)

print(f"A0 setup runtime:                    {a0_time:.3f} s")
print("A0 breakdown:")
print(f"  WDM grid construction:             {geom_precomp['wdm_grid_build_s']:.3f} s")
print(f"  one-arm plan construction:         {geom_precomp['one_arm_plan_build_s']:.3f} s")
print(f"  recursive TDI delay fields:        {geom_precomp['tdi_delay_build_s']:.3f} s")
print(f"  prepared 42-job TDI plan:          {geom_precomp['tdi_plan_build_s']:.3f} s")
print(f"  TDI delays loaded from cache:      {geom_precomp['tdi_delays_from_cache']}")
print(f"  other A0 overhead:                 {a0_time - a0_accounted:.3f} s")
print()
print(f"One-arm backend:                     {one_arm_shift_plan.resolved_assembly_backend}")
print(f"One-arm execution mode:              {wave_precomp['one_arm_execution_mode']}")
print(f"Parallel plus/cross DWTs:            {wave_precomp['parallel_wdm_transforms']}")
print(
    "One-arm parallel link schedule:      "
    f"{6 // wave_precomp['one_arm_parallel_links_per_group']} groups × "
    f"{wave_precomp['one_arm_parallel_links_per_group']} links"
)
print(
    "One-arm parallel max workers:        "
    f"{wave_precomp['one_arm_parallel_max_workers']}"
)
print(f"TDI backend:                         {tdi_plan.shift_plan.resolved_assembly_backend}")
print(f"One-arm persistent kernel memory:    {one_arm_shift_plan.kernel_memory_bytes / 1024**2:.2f} MiB")
print(f"TDI persistent kernel memory:        {tdi_plan.kernel_memory_bytes / 1024**2:.2f} MiB")
print(f"Process RSS after A0:                 {rss_after_a0:.2f} GiB")
print()
print(f"A1 synchronized waveform + 6 links:  {a1_time:.3f} s")
print(f"  waveform context / two DWTs:       {waveform_context_s:.3f} s")
print(f"  one-arm link stage + wrapper:       {synchronized_link_stage_s:.3f} s")
print(f"Process RSS after A1:                 {rss_after_a1:.2f} GiB")
print(f"One-arm coefficient shape:           {w_12_wdm.shape}")
print(f"One-arm coefficient dtype:           {w_12_wdm.dtype}")
print("One-arm links returned on device:    yes")
print("One-arm plan reused by A1:           yes")
print("TDI plan reused by A1:               yes")



### Second-generation Michelson TDI


In [ ]:
if RUN_TD_REFERENCE:
    t_tdi_td0 = time.perf_counter()

    X_td, Y_td, Z_td = gen2_michelson_td(
        times_pad,
        hplus_td,
        hcross_td,
        theta,
        phi,
        orbits_trailing,
        order=7,
        num_int_points=num_int_points,
    )

    td_tdi_time = time.perf_counter() - t_tdi_td0

    print(f"TD TDI runtime: {td_tdi_time:.3f} s")
    print(f"TD X/Y/Z shapes: {X_td.shape}, {Y_td.shape}, {Z_td.shape}")
else:
    X_td = Y_td = Z_td = None
    print("TD TDI reference skipped. Set RUN_TD_REFERENCE=True to run it.")


In [ ]:
# --------------------
# B: six prepared links -> X, Y, Z
# --------------------
# The prepared-plan path now uses grouped indexed accumulation.
# The first call may include JAX compilation for this shape.
t_b_cold = time.perf_counter()
X_wdm_cold, Y_wdm_cold, Z_wdm_cold = gen2_michelson_wdm_from_waveform_precomputed(
    wave_precomp,
    tdi_assembly_mode="prepared_plan",
)
block_result((X_wdm_cold, Y_wdm_cold, Z_wdm_cold))
b_cold_time = time.perf_counter() - t_b_cold

# Steady-state application of the already prepared and compiled plan.
t_b_warm = time.perf_counter()
X_wdm, Y_wdm, Z_wdm = gen2_michelson_wdm_from_waveform_precomputed(
    wave_precomp,
    tdi_assembly_mode="prepared_plan",
)
block_result((X_wdm, Y_wdm, Z_wdm))
b_warm_time = time.perf_counter() - t_b_warm
rss_after_grouped_b = rss_gib()

np.testing.assert_allclose(X_wdm, X_wdm_cold, rtol=1e-7, atol=1e-7)
np.testing.assert_allclose(Y_wdm, Y_wdm_cold, rtol=1e-7, atol=1e-7)
np.testing.assert_allclose(Z_wdm, Z_wdm_cold, rtol=1e-7, atol=1e-7)

print(f"B TDI cold runtime:                  {b_cold_time:.3f} s")
print(f"B TDI warm runtime:                  {b_warm_time:.3f} s")
print(f"Process RSS after grouped B:          {rss_after_grouped_b:.2f} GiB")
print(f"WDM X/Y/Z coefficient shapes:        {X_wdm.shape}, {Y_wdm.shape}, {Z_wdm.shape}")

print("\nStaged WDM timing:")
print(f"A0 static setup:                     {a0_time:.3f} s")
print(f"A1 waveform + 6 links:               {a1_time:.3f} s")
print(f"B grouped TDI warm application:      {b_warm_time:.3f} s")
print(f"A1 + B repeated cost:                {a1_time + b_warm_time:.3f} s")



### Comparison of the TD and WDM methods in WDM space

The TD references are transformed with the **same WDM object** used by the staged response calculation. Boundary and \(m=0\) cuts are applied through the utility helpers.


In [ ]:
if RUN_TD_REFERENCE:
    # The waveform generator has already padded times_pad to a valid WDM length,
    # so use the exact same transform object as the response pipeline.
    if len(dl_12_td) != int(wdm_response.N):
        raise ValueError(
            f"TD response length {len(dl_12_td)} does not match WDM N={wdm_response.N}."
        )

    w_12_td = wdm_response.dwt(dl_12_td)

    cut_info_one_arm = compute_wdm_cut_info(
        original_length=N,
        Nf=Nf,
        dt=dt,
        alpha=alpha,
        tukey_factor=tukey_factor,
        window_pad_left=pad_info["window_pad_left"],
        window_pad_right=pad_info["window_pad_right"],
        wdm_dT=wdm_response.dT,
        wdm_Nt=wdm_response.Nt,
        include_wdm_tail_padding=True,
        include_taper_margin=True,
        drop_m0=True,
    )

    f_max = wdm_response.Nf * wdm_response.dF

    fig, ax = WDM.plotting.time_frequency_plot(
        wdm_response, w_12_td, part="real", scale="linear"
    )
    ax.set_ylim(0, f_max)
    plt.show()

    fig, ax = WDM.plotting.time_frequency_plot(
        wdm_response, w_12_wdm, part="real", scale="linear"
    )
    ax.set_ylim(0, f_max)
    plt.show()

    fig, ax = WDM.plotting.time_frequency_plot(
        wdm_response, w_12_td - w_12_wdm, part="abs", scale="linear"
    )
    ax.set_ylim(0, f_max)
    plt.show()

    td_cut = apply_wdm_cut(w_12_td, cut_info_one_arm)
    wdm_cut = apply_wdm_cut(np.asarray(w_12_wdm).real, cut_info_one_arm)

    ov_wdm, mm_wdm = wdm_mismatch(td_cut, wdm_cut)
    print(f"One-arm WDM-domain overlap={ov_wdm:.9f}, mismatch={mm_wdm:.3e}")
    print(
        "Applied cuts:",
        f"left={cut_info_one_arm['left_cut']},",
        f"right={cut_info_one_arm['right_cut']},",
        f"drop_m0={cut_info_one_arm['trim_m0']}",
    )
else:
    cut_info_one_arm = None
    print("One-arm TD-vs-WDM comparison skipped.")


In [ ]:
if RUN_TD_REFERENCE:
    # TDI response plot in the time domain (TD baseline)
    plt.figure(figsize=(10, 5))
    plt.plot(
        plotting_times[plt_left_lim:plt_left_lim + 4000],
        X_td[plt_left_lim:plt_left_lim + 4000],
        label="X_td",
    )
    plt.plot(
        plotting_times[plt_left_lim:plt_left_lim + 4000],
        Y_td[plt_left_lim:plt_left_lim + 4000],
        label="Y_td",
    )
    plt.plot(
        plotting_times[plt_left_lim:plt_left_lim + 4000],
        Z_td[plt_left_lim:plt_left_lim + 4000],
        label="Z_td",
    )
    plt.title("Gen2 Michelson TDI responses")
    plt.xlabel("time [s]")
    plt.ylabel("response")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    if not (len(X_td) == len(Y_td) == len(Z_td) == int(wdm_response.N)):
        raise ValueError("TD TDI arrays do not match the staged WDM grid length.")

    X_td_w = wdm_response.dwt(X_td)
    Y_td_w = wdm_response.dwt(Y_td)
    Z_td_w = wdm_response.dwt(Z_td)

    cut_info_tdi = cut_info_one_arm

    X_td_cut = apply_wdm_cut(X_td_w, cut_info_tdi)
    Y_td_cut = apply_wdm_cut(Y_td_w, cut_info_tdi)
    Z_td_cut = apply_wdm_cut(Z_td_w, cut_info_tdi)

    X_wdm_cut = apply_wdm_cut(np.asarray(X_wdm).real, cut_info_tdi)
    Y_wdm_cut = apply_wdm_cut(np.asarray(Y_wdm).real, cut_info_tdi)
    Z_wdm_cut = apply_wdm_cut(np.asarray(Z_wdm).real, cut_info_tdi)

    X_ov, X_mm = wdm_mismatch(X_td_cut, X_wdm_cut)
    Y_ov, Y_mm = wdm_mismatch(Y_td_cut, Y_wdm_cut)
    Z_ov, Z_mm = wdm_mismatch(Z_td_cut, Z_wdm_cut)

    print("TDI TD-vs-WDM mismatch in WDM domain:")
    print(f"X: overlap={X_ov:.9f}, mismatch={X_mm:.3e}")
    print(f"Y: overlap={Y_ov:.9f}, mismatch={Y_mm:.3e}")
    print(f"Z: overlap={Z_ov:.9f}, mismatch={Z_mm:.3e}")

    plt.figure(figsize=(10, 4))
    plt.plot(np.abs(X_td_cut - X_wdm_cut).mean(axis=1), label="X")
    plt.plot(np.abs(Y_td_cut - Y_wdm_cut).mean(axis=1), label="Y")
    plt.plot(np.abs(Z_td_cut - Z_wdm_cut).mean(axis=1), label="Z")
    plt.title("Mean absolute coefficient difference per retained time row")
    plt.xlabel("retained WDM time-row index")
    plt.ylabel("mean |delta coefficient|")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()
else:
    print("TDI TD-vs-WDM comparison skipped.")



### One-arm production-route validation

The selected production A1 route uses parallel plus/cross DWTs and two
concurrent groups of three directed-link shifts. It is checked against:

1. the ordinary device-materialized six-link route;
2. the host materialized reference;
3. the optional fused-low-memory device route.

All comparisons use the same prepared one-arm operator.


In [ ]:

if RUN_ONE_ARM_VALIDATION:
    # ------------------------------------------------------------
    # 1. Ordinary device-materialized route
    # ------------------------------------------------------------
    wave_precomp_ordinary = precompute_gen2_michelson_wdm_waveform(
        geom_precomp,
        hplus_td,
        hcross_td,
        return_device_links=True,
        parallel_wdm_transforms=PARALLEL_WDM_TRANSFORMS,
        one_arm_execution_mode="materialized",
    )

    production_links = wave_precomp["link_w"]
    ordinary_links = wave_precomp_ordinary["link_w"]

    block_result(production_links)
    block_result(ordinary_links)

    production_vs_ordinary_max_abs = 0.0
    production_vs_ordinary_max_rel_l2 = 0.0

    for link in ordinary_links:
        production_value = np.asarray(production_links[link])
        ordinary_value = np.asarray(ordinary_links[link])
        difference = production_value - ordinary_value
        ordinary_norm = np.linalg.norm(ordinary_value)

        relative_l2 = (
            float(np.linalg.norm(difference) / ordinary_norm)
            if ordinary_norm > 0
            else float(np.linalg.norm(difference))
        )

        production_vs_ordinary_max_abs = max(
            production_vs_ordinary_max_abs,
            float(np.max(np.abs(difference))),
        )
        production_vs_ordinary_max_rel_l2 = max(
            production_vs_ordinary_max_rel_l2,
            relative_l2,
        )

        np.testing.assert_allclose(
            production_value,
            ordinary_value,
            rtol=0.0,
            atol=0.0,
        )

    # ------------------------------------------------------------
    # 2. Host materialized reference
    # ------------------------------------------------------------
    wave_precomp_host = precompute_gen2_michelson_wdm_waveform(
        geom_precomp,
        hplus_td,
        hcross_td,
        return_device_links=False,
        parallel_wdm_transforms=False,
    )
    host_links = wave_precomp_host["link_w"]

    production_vs_host_max_abs = 0.0
    production_vs_host_max_rel_l2 = 0.0

    for link in host_links:
        production_value = np.asarray(production_links[link])
        host_value = np.asarray(host_links[link])
        difference = production_value - host_value
        host_norm = np.linalg.norm(host_value)

        relative_l2 = (
            float(np.linalg.norm(difference) / host_norm)
            if host_norm > 0
            else float(np.linalg.norm(difference))
        )

        production_vs_host_max_abs = max(
            production_vs_host_max_abs,
            float(np.max(np.abs(difference))),
        )
        production_vs_host_max_rel_l2 = max(
            production_vs_host_max_rel_l2,
            relative_l2,
        )

        np.testing.assert_allclose(
            production_value,
            host_value,
            rtol=2e-6,
            atol=2e-7,
        )
        assert isinstance(production_links[link], jax.Array)

    # ------------------------------------------------------------
    # 3. Optional fused-low-memory device route
    # ------------------------------------------------------------
    wave_precomp_fused = precompute_gen2_michelson_wdm_waveform(
        geom_precomp,
        hplus_td,
        hcross_td,
        return_device_links=True,
        parallel_wdm_transforms=PARALLEL_WDM_TRANSFORMS,
        one_arm_execution_mode="fused_low_memory",
    )

    fused_links = wave_precomp_fused["link_w"]
    block_result(fused_links)

    fused_max_abs = 0.0
    fused_max_rel_l2 = 0.0

    for link in production_links:
        production_value = np.asarray(production_links[link])
        fused_value = np.asarray(fused_links[link])
        difference = fused_value - production_value
        production_norm = np.linalg.norm(production_value)

        relative_l2 = (
            float(np.linalg.norm(difference) / production_norm)
            if production_norm > 0
            else float(np.linalg.norm(difference))
        )

        fused_max_abs = max(
            fused_max_abs,
            float(np.max(np.abs(difference))),
        )
        fused_max_rel_l2 = max(
            fused_max_rel_l2,
            relative_l2,
        )

        np.testing.assert_allclose(
            fused_value,
            production_value,
            rtol=2e-6,
            atol=2e-7,
        )

    print("One-arm production-route validation:")
    print(
        "  parallel vs ordinary max abs:      "
        f"{production_vs_ordinary_max_abs:.6e}"
    )
    print(
        "  parallel vs ordinary max rel L2:   "
        f"{production_vs_ordinary_max_rel_l2:.6e}"
    )
    print(
        "  parallel vs host max abs:          "
        f"{production_vs_host_max_abs:.6e}"
    )
    print(
        "  parallel vs host max rel L2:       "
        f"{production_vs_host_max_rel_l2:.6e}"
    )
    print(
        "  fused vs parallel max abs:         "
        f"{fused_max_abs:.6e}"
    )
    print(
        "  fused vs parallel max rel L2:      "
        f"{fused_max_rel_l2:.6e}"
    )
    print(
        "  device link dtype:                 "
        f"{production_links[(1, 2)].dtype}"
    )

    del (
        wave_precomp_ordinary,
        wave_precomp_host,
        wave_precomp_fused,
        ordinary_links,
        host_links,
        fused_links,
    )
    gc.collect()
else:
    print("One-arm validation skipped.")


In [ ]:

# --------------------
# Same-backend one-arm link-stage benchmark
# --------------------
# The plus/cross DWTs are performed once outside the timed loop.
if RUN_ONE_ARM_BENCHMARK:
    from WDM_LISAresponse.code.lisa_response.response_with_plan import (
        _build_wdm_dT_context_from_wdm,
    )

    one_arm_plan = geom_precomp["one_arm_plan"]

    wave_context_one_arm = _build_wdm_dT_context_from_wdm(
        geom_precomp["times"],
        hplus_td,
        hcross_td,
        geom_precomp["wdm"],
        m_offset=0.0,
        parallel_wdm_transforms=PARALLEL_WDM_TRANSFORMS,
    )

    w_t_plus = wave_context_one_arm["w_t_plus"]
    w_t_cross = wave_context_one_arm["w_t_cross"]

    def run_ordinary_materialized():
        return one_arm_plan.apply_device(
            w_t_plus,
            w_t_cross,
            execution_mode="materialized",
        )

    def run_parallel_materialized():
        return one_arm_plan.apply_device(
            w_t_plus,
            w_t_cross,
            execution_mode="materialized_parallel_links",
            parallel_links_per_group=(
                ONE_ARM_PARALLEL_LINKS_PER_GROUP
            ),
            parallel_max_workers=(
                ONE_ARM_PARALLEL_MAX_WORKERS
            ),
        )

    def run_fused_low_memory():
        return one_arm_plan.apply_device(
            w_t_plus,
            w_t_cross,
            execution_mode="fused_low_memory",
        )

    routes = {
        "ordinary_materialized": run_ordinary_materialized,
        "parallel_2x3_materialized": run_parallel_materialized,
        "fused_low_memory": run_fused_low_memory,
    }

    warm_results = {
        name: function()
        for name, function in routes.items()
    }
    block_result(warm_results)

    reference_value = np.asarray(
        warm_results["ordinary_materialized"]
    )
    for name, result in warm_results.items():
        np.testing.assert_allclose(
            np.asarray(result),
            reference_value,
            rtol=2e-6,
            atol=2e-7,
        )
    del warm_results, reference_value

    timings = {
        name: []
        for name in routes
    }
    route_names = list(routes)

    for repeat_index in range(ONE_ARM_BENCHMARK_REPEATS):
        rotation = repeat_index % len(route_names)
        order = (
            route_names[rotation:]
            + route_names[:rotation]
        )

        for name in order:
            started = time.perf_counter()
            result = routes[name]()
            block_result(result)
            timings[name].append(
                time.perf_counter() - started
            )
            del result

    print(
        f"One-arm link-stage benchmark "
        f"({ONE_ARM_BENCHMARK_REPEATS} runs per route):"
    )

    medians = {}
    for name in route_names:
        values = timings[name]
        medians[name] = float(np.median(values))
        q25, q75 = np.percentile(values, [25.0, 75.0])

        print()
        print(f"  {name}:")
        print(f"    median:  {medians[name]:.3f} s")
        print(f"    q25:     {q25:.3f} s")
        print(f"    q75:     {q75:.3f} s")
        print(
            "    all:    ",
            [round(value, 3) for value in values],
        )

    print()
    print(
        "  parallel runtime change: "
        f"{100.0 * (medians['parallel_2x3_materialized'] / medians['ordinary_materialized'] - 1.0):+.1f}%"
    )
    print(
        "  fused runtime change:    "
        f"{100.0 * (medians['fused_low_memory'] / medians['ordinary_materialized'] - 1.0):+.1f}%"
    )

    gc.collect()
else:
    print("One-arm benchmark skipped.")


### Long-duration grouped-versus-materialized TDI validation

The grouped path remains the normal laptop execution mode. The materialized
route is optional because at \(N_t=32640\), \(N_f=100\), its 42-job output is
approximately \(1.05\) GiB. Enable `RUN_MATERIALIZED_REFERENCE` only when an
explicit numerical or scaling comparison is required.


In [ ]:
# Always benchmark the long-duration grouped path.
grouped_median = median_runtime(
    lambda: tdi_plan.apply(link_w),
    repeats=BENCHMARK_REPEATS,
)

print("Prepared TDI execution:")
print(f"  grouped median:       {grouped_median:.3f} s")

if RUN_MATERIALIZED_REFERENCE:
    # Numerical agreement against the old fully materialized pathway.
    grouped_reference = tdi_plan.apply(link_w)
    materialized_reference = tdi_plan.apply_materialized(link_w)

    for name, grouped_channel, materialized_channel in zip(
        ("X", "Y", "Z"),
        grouped_reference,
        materialized_reference,
    ):
        np.testing.assert_allclose(
            grouped_channel,
            materialized_channel,
            rtol=2e-5,
            atol=2e-6,
        )
        print(f"  {name}: grouped path matches materialized reference")

    # Benchmark the materialized route only when explicitly requested.
    materialized_median = median_runtime(
        lambda: tdi_plan.apply_materialized(link_w),
        repeats=BENCHMARK_REPEATS,
    )

    print(f"  materialized median:  {materialized_median:.3f} s")
    print(
        f"  materialized/grouped: "
        f"{materialized_median / grouped_median:.2f}x"
    )
else:
    materialized_median = np.nan
    print("  materialized route:   skipped")


In [ ]:
(_, _, _), grouped_profile = tdi_plan.apply(
    link_w,
    return_profile=True,
)

print("\nGrouped TDI profile:")
print(f"  execution mode:               {grouped_profile['tdi_execution_mode']}")
print(f"  JAX backend:                  {jax.default_backend()}")
print(f"  distinct source arrays:       {grouped_profile['n_sources']}")
print(f"  shift jobs:                   {grouped_profile['n_jobs']}")
print(f"  final output arrays:          {grouped_profile['n_outputs']}")
print(
    f"  full materialized output:     "
    f"{grouped_profile['full_materialized_output_bytes'] / 1024**2:.2f} MiB"
)
print(
    f"  actual returned output:       "
    f"{grouped_profile['returned_output_bytes'] / 1024**2:.2f} MiB"
)
print(
    f"  avoided materialized output:  "
    f"{grouped_profile['host_transfer_bytes_saved'] / 1024**2:.2f} MiB"
)
print(
    f"  cached device-plan storage:   "
    f"{grouped_profile['device_plan_memory_bytes'] / 1024**2:.2f} MiB"
)
print(f"  current process RSS:          {rss_gib():.2f} GiB")


### Historical one-year-scale baseline

For the earlier CPU/JAX implementation with \(N=3{,}200{,}000\),
\(N_t=32640\), \(N_f=100\), complex64 assembly, and
\(L_{
m trunc}=25\):

- reusable \(A_0\): approximately **21.4 s**;
- old \(A_1\): approximately **14.4 s**;
- grouped \(B\): approximately **27.9–29.7 s**;
- materialized \(B\): approximately **52.7 s**;
- avoided 42-job materialization: approximately **971 MiB**;
- expanded persistent TDI plan: approximately **2.16 GiB**.

These values are retained only as a historical reference. Use the synchronized
timings printed by the current notebook for the device-materialized one-arm
production baseline. The halo-fused route is now treated as an optional
low-memory comparison.



### Repeated production A1 benchmark

This final check times the complete waveform-dependent one-arm stage through
the normal staged API. It compares the former sequential-DWT/ordinary-shift
route with the selected parallel-DWT/two-groups-of-three route.


In [ ]:

# --------------------
# Complete A1 production benchmark
# --------------------
if RUN_ONE_ARM_BENCHMARK:
    def run_ordinary_a1():
        return precompute_gen2_michelson_wdm_waveform(
            geom_precomp,
            hplus_td,
            hcross_td,
            return_device_links=True,
            parallel_wdm_transforms=False,
            one_arm_execution_mode="materialized",
        )

    def run_parallel_a1():
        return precompute_gen2_michelson_wdm_waveform(
            geom_precomp,
            hplus_td,
            hcross_td,
            return_device_links=True,
            parallel_wdm_transforms=PARALLEL_WDM_TRANSFORMS,
            one_arm_execution_mode=ONE_ARM_EXECUTION_MODE,
            one_arm_parallel_links_per_group=(
                ONE_ARM_PARALLEL_LINKS_PER_GROUP
            ),
            one_arm_parallel_max_workers=(
                ONE_ARM_PARALLEL_MAX_WORKERS
            ),
        )

    # Warm both complete staged routes.
    ordinary_result = run_ordinary_a1()
    parallel_result = run_parallel_a1()
    block_result(ordinary_result)
    block_result(parallel_result)

    for link in ordinary_result["link_w"]:
        np.testing.assert_allclose(
            np.asarray(parallel_result["link_w"][link]),
            np.asarray(ordinary_result["link_w"][link]),
            rtol=0.0,
            atol=0.0,
        )

    ordinary_times = []
    parallel_times = []

    for repeat_index in range(FULL_A1_BENCHMARK_REPEATS):
        order = (
            ("ordinary", "parallel")
            if repeat_index % 2 == 0
            else ("parallel", "ordinary")
        )

        for route in order:
            started = time.perf_counter()
            result = (
                run_ordinary_a1()
                if route == "ordinary"
                else run_parallel_a1()
            )
            block_result(result)
            elapsed = time.perf_counter() - started

            if route == "ordinary":
                ordinary_times.append(elapsed)
            else:
                parallel_times.append(elapsed)

            del result

    ordinary_median = float(np.median(ordinary_times))
    parallel_median = float(np.median(parallel_times))

    print(
        f"Complete A1 benchmark "
        f"({FULL_A1_BENCHMARK_REPEATS} runs per route):"
    )
    print(
        "  ordinary sequential-DWT/materialized median: "
        f"{ordinary_median:.3f} s"
    )
    print(
        "  selected parallel production median:        "
        f"{parallel_median:.3f} s"
    )
    print(
        "  runtime change:                              "
        f"{100.0 * (parallel_median / ordinary_median - 1.0):+.1f}%"
    )
    print(
        "  ordinary all:",
        [round(value, 3) for value in ordinary_times],
    )
    print(
        "  parallel all:",
        [round(value, 3) for value in parallel_times],
    )

    del ordinary_result, parallel_result
    gc.collect()
else:
    print("Complete A1 benchmark skipped.")
